In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer

# Imputation Functions

In [2]:
def impute_app_inputs(df) -> pd.DataFrame:
    """
    Impute application inputs (canopy_moisture and surface_moisture) using mean
    No prerequisites
    """
    na_cols = ["canopy_moisture", "surface_moisture"]
    for col in na_cols:
        na_indices = df[col].isna().index
        df.loc[na_indices, col] = df[col].mean()
    return df

def impute_cpu_request(df) -> pd.DataFrame:
    """
    Impute cpu request using cpu usage logic (e.g. max of 300% means cpu_request == 3)
    Find the max value in each row's f"cpu_usage_%_t{i}" columns.
    For each missing cpu request, replace it with the ceil of that row's max usage_% (or 1)
    Prerequisites:
        1. CPU feature manipulation
    """
    cpu_cols = [col for col in df if "cpu_usage" in col]
    row_max = df[cpu_cols].max(axis=1)
    fill_values = row_max.where(row_max > 0, 1)
    df["cpu_request"] = df["cpu_request"].fillna(fill_values.apply(np.ceil))
    return df

def impute_queried_data(df) -> pd.DataFrame:
    """
    Impute queried data †hat is -2 or -1 using KNN with mean.
    Does not impute cpu_request.
    Prerequisites:
        1. Add train col
        2. All feature manipulations
        3. impute CPU request
        4. log transformations
        5. scaling
    """
    t_i_cols = [col for col in df.columns if col[-3:-1]=="_t"]
    cols_to_impute = ["mem_request"] + t_i_cols
    imputer = KNNImputer(missing_values=-1, n_neighbors=3)
    # get all missing values as the same value
    df[cols_to_impute] = df[cols_to_impute].clip(lower=-1)
    # only fit KNN on training data
    X_train = df[df["train"] == True].to_numpy()
    imputer.fit(X_train)
    # transform entire df based on fit from training
    imputed_array = imputer.transform(df.to_numpy())
    imputed_df = pd.DataFrame(imputed_array, columns=df.columns)
    return imputed_df
    


# Training Split Functions

In [3]:
def add_train_col(df:pd.DataFrame, test_frac:float=0.2) -> pd.DataFrame:
    """
    Return a new dataframe with a boolean 'train' column added to the original dataframe.
    Use stratification to ensure a more even distribution of runtime lengths.
    No prerequisites
    """
    bins = [0, 300, 600, 3600, float("inf")]
    labels = ["0-5m", "5-10m", "10-60m", "60m+"]
    runtime_bins = pd.cut(
        df["runtime"],
        bins=bins,
        labels=labels,
        right=False
    )
    X = df[[c for c in df.columns if c != "runtime"]]
    y = df["runtime"]
    X_train, X_test, _, _ = train_test_split(
        X,
        y,
        test_size=test_frac,
        stratify=runtime_bins,
        random_state=42
    )
    df["train"] = True
    df.loc[X_test.index, "train"] = False
    assert set(df[df["train"] == True].index) == set(X_train.index), "train indices don't match"
    assert set(df[df["train"] == False].index) == set(X_test.index), "test indices don't match"
    return df

def add_time_sample_col(df:pd.DataFrame) -> pd.DataFrame:
    """
    Add a time sample col with values 0 through 4 to indicate which bin the run was apart of
    [0,1m)      = 0
    [1m, 5m)    = 1
    [5m, 10m)   = 2
    [10m, 1h)   = 3
    1h+         = 4
    """
    duration_minutes = [1, 5, 10, 60]
    bins = [minutes * 60 for minutes in duration_minutes]
    df["time_sample"] = np.digitize(df["runtime"], bins, right=False)
    return df


# Transformation and Feature Manipulation Functions

In [4]:
def log_transform(df: pd.DataFrame) -> pd.DataFrame:
    """
    Log transform relevant columns (runtime + t_i performance metrics)
    Prerequisites:
        1. Add train col
        2. All feature manipulations
        3. Add time sample col
    """
    # mem_request & cpu_request only take a few values and did not seem to need log transformations.
    queried_t_cols = [col for col in df.columns if col[-3:-1] == "_t"]
    transform_cols = [col for col in queried_t_cols if "_%" not in col]  # don't transform cpu %
    log_transform_cols = ["runtime"] + transform_cols
    for col in log_transform_cols:
        positive = df[col] > 0
        df.loc[positive, col] = np.log(df[col][positive])
        df.loc[~positive, col] = -1
    # change column names to show they've been log transformed
    name_change = {col: "log_"+col for col in log_transform_cols}
    df = df.rename(columns=name_change)
    return df


def network_feature_manipulation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Combine transmitted and received packets as well as bandwidth
    Run this before any imputations
    """
    def combine(row):
        """
        Combine 2 values for network usage. Deal with missing values
        If both are missing, return -1. If one is missing, return double the other.
        Usually they're roughly the same, so doubling is a good strategy
        """
        val1, val2 = row.iloc[0], row.iloc[1]
        val1_missing = val1 == -1
        val2_missing = val2 == -1
        if not (val1_missing or val2_missing):
            return val1 + val2
        if val1_missing and val2_missing:
            return -1
        if val1_missing:
            return 2 * val2
        if val2_missing:
            return 2 * val1

    drop_cols = []
    for i in range(1, 5):
        t_i = f"t{i}"
        # combine transmitted and received packets
        transmitted_col = f"transmitted_packets_{t_i}"
        received_col = f"received_packets_{t_i}"
        df[f"total_packets_{t_i}"] = df[[transmitted_col, received_col]].apply(combine, axis=1)
        drop_cols += [transmitted_col, received_col]
        # combine transmitted and received bandwidth
        transmitted_col = f"transmitted_bandwidth_{t_i}"
        received_col = f"received_bandwidth_{t_i}"
        df[f"total_bandwidth_{t_i}"] = df[[transmitted_col, received_col]].apply(combine, axis=1)
        drop_cols += [transmitted_col, received_col]
    df = df.drop(columns=drop_cols)
    return df

def cpu_feature_manipulation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Get cpu usage as a percentage rather than cpu_seconds, then drop duration cols
    Run this before any imputations
    """
    cpu_usage_cols = [col for col in df.columns if "cpu_usage_t" in col]
    for col in cpu_usage_cols:
        i = int(col[-1])  # cols are of the form f"cpu_usage_t{i}" -> extract i. Ex: cpu_usage_t2
        metric = col[:-3]  # everything until f"_t{i}"
        suffix = col[-3:]  # f"_t{i}"
        percent_col = metric + "_%" + suffix
        df[percent_col] = df[col] / df[f"duration_t{i}"]

    drop_cols = cpu_usage_cols + [col for col in df.columns if "duration_t" in col]
    df = df.drop(columns=drop_cols)
    return df

def cpu_mem_feature_additions(df: pd.DataFrame) -> pd.DataFrame:
    for i in range(1,5):
        # cpu efficiency
        cpu_col = f"cpu_usage_%_t{i}"
        df[f"cpu_efficiency_t{i}"] = df[cpu_col]
        # memory growth
        mem_col = f"mem_usage_t{i}"
        prev = f"mem_usage_t{i-1}"
        if prev in df.columns:
            df[f"mem_growth_t{i}"] = df[mem_col] - df[prev]
            df[f"mem_growth_t{i}"] = df[f"mem_growth_t{i}"].apply(lambda val: np.clip(val, -1, None))
    return df


# Scaling

Certain machine learning algorithms become less accurate if the data has a large variety of scales between features. This is not the case with tree-based algorithms, but for something like clustering, scaling each feature to a given range is going to help treat all features with the same initial importance, rather than deriving importance from the scale of the features. This should also help speed up convergence for linear models and neural networks. Scaling should be done based on the training set, rather than the entire dataset to avoid data leakage. That said, the test set should still be scaled, just with the criteria from the train set.

In [5]:
def scale(df: pd.DataFrame, log_transformed_runtime:bool = True) -> pd.DataFrame:
    """
    Scale all numeric columns (except runtime) to be between 0 and 1
    Assumes 'train' is the only non-numeric column
    Prerequisites:
        1. Add train col
        2. All feature manipulations
        3. Add time sample col
        4. log transformations
    """
    scaler = MinMaxScaler(feature_range=(0,1))
    runtime_col = "log_runtime" if log_transformed_runtime else "runtime"
    # only fit on X_train, then scale everything else with that
    X_train = df[df["train"] == True].drop(columns=["train", "time_sample", runtime_col]).to_numpy()
    scaler.fit(X_train)
    
    numeric_df = df.drop(columns=["train", "time_sample", runtime_col])
    scaled_array = scaler.transform(numeric_df.to_numpy())
    
    scaled_df = pd.DataFrame(scaled_array, columns=numeric_df.columns, index=df.index)
    scaled_df["train"] = df["train"]
    scaled_df["time_sample"] = df["time_sample"]
    scaled_df[runtime_col] = df[runtime_col]
    return scaled_df


# Running the code

In [6]:
df = pd.read_csv("sanitized_data/data.csv")

The EDA identified a few columns that we would not need. Let's drop those.

In [7]:
useless_cols = [
    "threads",          # all datapoints have threads=1 since BP3D is not concurrent
    "wind_direction",   # missing necessary information on how this will impact the fire simulation
    "queue_seconds"     # noise from waiting on shared distributed system
]

# total preformance data columns get collected at the end of the run
data_leak_cols = [
    "run_max_mem_rss_bytes",
    "sim_time",
    "transmitted_packets_total",
    "received_packets_total",
    "transmitted_bandwidth_total",
    "received_bandwidth_total",
    "cpu_usage_total",
    "mem_usage_total"
]

drop_cols = useless_cols + data_leak_cols
df = df.drop(columns=drop_cols, errors='ignore')

Now let's run the functions we've defined earlier. 

First we make sure there is a stratified mix of runtimes of all different lengths. 

Then we calculate the cpu_usage_% as cpu_seconds / sample time. This means it can be over 1.00 if it is running on several CPUs.

Next, we define total network metrics as received + transmitted.

From there, we impute the simple metrics that don't rely on KNN.

Then we log transform the relevant data to improve the distributions and make them less exponential.

After that, we scale the predictors to be between 0 and 1. This makes linear models, neural nets, and regularrization models improve in speed and accuracy.

Finally, we impute the more complex columns with KNN.

In [8]:
df = add_train_col(df)
df = add_time_sample_col(df)
df = cpu_feature_manipulation(df)
df = network_feature_manipulation(df)
df = impute_cpu_request(df)
df = impute_app_inputs(df)
# df = cpu_mem_feature_additions(df)  # these did not change results --> no need to add it
df = log_transform(df)
df = scale(df)
df = impute_queried_data(df)

In [9]:
df

,canopy_moisture,surface_moisture,wind_speed,area,cpu_request,mem_request,log_mem_usage_t1,log_mem_usage_t2,log_mem_usage_t3,log_mem_usage_t4,...,log_total_bandwidth_t1,log_total_packets_t2,log_total_bandwidth_t2,log_total_packets_t3,log_total_bandwidth_t3,log_total_packets_t4,log_total_bandwidth_t4,train,time_sample,log_runtime
0,0.0,0.0,0.055556,0.002823,0.333333,1.455192e-11,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,1.0,5.371908
1,0.0,0.0,0.055556,0.002823,0.333333,1.455192e-11,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.0,5.312495
2,0.0,0.0,0.055556,0.002823,0.333333,1.455192e-11,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.0,5.398052
3,0.0,0.0,0.049667,0.114324,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,3.0,6.661027
4,0.0,0.0,0.077778,0.036334,1.000000,2.500000e-01,0.814427,0.815957,0.816412,0.819619,...,0.795133,0.597933,0.767203,0.601408,0.767153,0.636934,0.773489,1.0,4.0,8.449182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20310,0.0,0.0,0.022222,0.023786,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.805473,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.854326,0.873339,1.0,4.0,9.462577
20311,0.0,0.0,0.111111,0.023786,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.805968,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.839998,0.861611,1.0,4.0,9.387817
20312,0.0,0.0,0.111111,0.023786,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.805968,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.840085,0.861665,1.0,4.0,9.547241
20313,0.0,0.0,0.022222,0.023786,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.805968,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.840009,0.861625,1.0,4.0,10.110339


In [10]:
df.to_csv("preprocessed_data/data.csv", index=False)